# Imports in a Python notebook
In this lab, you will mainly implement classes that let you build neural networks. These classes will be stored in a file `Neural.py`, which already contains the `Generic` class. However, a notebook’s default behavior when you import a module is **not** to reload the file, so changes you make in `Neural.py` will not be picked up. To reload the module when the file changes, run the following commands:

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import Neural as Neur

In [2]:
import Neural as Neur
L=Neur.Generic()
print(L.backward(1))
# You should get
# (None, None)

(None, None)


Copy and paste the `Generic` class into a class `Arctan` and check whether Python reloads the `Neural` file when you run the `import` command.


In [3]:
import Neural as Neur

L=Neur.Arctan()
print(L.backward(1))
# You should get
# (None, None)

(None, None)


# Defining a layer
The file `Neural.py` will be a library we use to build our neural networks; we will gradually add different layer types.
By default, all layers share at least the same methods and attributes as the `Generic` layer, and may define more. Mathematically, recall that a layer is a function $F$ that takes data $X$ and returns a vector $Y$. Internally, the layer has **parameters** that are meant to be **learned**. Parameters are denoted $\theta$. Thus
$$ Y=F(\theta,X)$$

The methods are:

* `set_params`: sets $\theta$, the layer’s internal parameters.
* `get_params`: returns $\theta$, the layer’s internal parameters.
* `forward`: applies $F$ to the data $X$ and internal parameters $\theta$, and returns $Y$.
* `backward`: applies gradient backpropagation to an output gradient `grad_output` and computes
  * the local gradient `grad_local`, i.e. the gradient with respect to the internal parameters
  * the input gradient `grad_entree`, i.e. the gradient with respect to $X$

Internal attributes include:
* `self.nb_params`: the length of the vector $\theta$.

Remember the following about vector shapes:
* $X$ and `grad_entree` have the same shape (`grad_entree` is the gradient with respect to $X$).
* $Y$ and `grad_output` have the same shape (`grad_output` is the gradient with respect to $Y$).
* $\theta$ and `grad_local` have the same length, namely `self.nb_params` (`grad_local` is the gradient with respect to $\theta$).


Layers automatically store the `X` passed to `forward`; it is kept in `self.save_X`. Saving `X` is useful because it is needed again in `backward`. This does, however, make the network memory-hungry. Other designs are possible, but we will stick to this one in the labs.

# Data layout

The data $X$, $Y$ and parameters $\theta$ are numpy arrays (with compatible shapes).


# Implementing the Arctan layer
We now turn to our first layer: you should have copied `Generic` into a class `Arctan`. We will fill in this class.

The `Arctan` layer takes an input vector $X$ of size $p$ and returns a vector $Y$ of size $p$ such that
$$Y[i]=\phi(X[i]) \quad \forall 1\le i\le p$$

where $\phi$ is the arctangent function. This layer has no local parameters (`self.nb_params=0`); `set_params` and `get_params` are no-ops. The backward pass is
$$ g_e[i]=\phi'(X[i])g_s[i] \quad \forall 1\le i\le p$$
where $g_e$ and $g_s$ are the input and output gradients, respectively. Recall that `X` was stored in `self.save_X`.

Implement this layer and run the code below; for the local gradient and for `get_params`, return `None`.

In [4]:
import Neural as Neur
L=Neur.Arctan()
np.random.seed(10)
X=np.random.randn(4,2)
grad_sortie=np.random.randn(4,2)
print('nb_params=',L.nb_params)
print('forward=',L.forward(X))
print('backward=',L.backward(grad_sortie))
# You should get
# nb_params= 0
# forward= [[ 0.92666583  0.62090688]
# [-0.99647548 -0.00838365]
# [ 0.55596017 -0.6240794 ]
# [ 0.25952369  0.10812518]]
# backward=  (None, array([[ 0.00154751, -0.11550505],
# [ 0.12780186,  1.20295282],
# [-0.69626624,  0.67715401],
# [ 0.21357394,  0.43995373]]))


nb_params= 0
forward= [[ 0.92666583  0.62090688]
 [-0.99647548 -0.00838365]
 [ 0.55596017 -0.6240794 ]
 [ 0.25952369  0.10812518]]
backward= (None, array([[ 0.00154751, -0.11550505],
       [ 0.12780186,  1.20295282],
       [-0.69626624,  0.67715401],
       [ 0.21357394,  0.43995373]]))


# Other layers
Following the Arctan layer pattern, you can build many layers by changing $\Phi$ (and of course $\Phi'$). These are all simple, parameter-free layers that only apply a nonlinearity to the data. Here is a table of a few common layers and their $\Phi$ and $\Phi'$:


| Name           |     $\Phi(X)$        |        $\Phi'(X)$                 |
| :------------  | :---------------:    | ----------------------:           |
| Sigmoid        | $$\frac 1 {1+e^{-X}}$$ | $$\frac {e^{-X}} {(1+e^{-X})^2}$$ |
| ReLU           |   $$\max(X,0)$$        |   $$\max(\frac{X}{|X|},0)$$          |
| ABS            |     $$\|X\|$$            |        $$\frac{X}{\|X\|}$$              |

# Data layout
Before we continue, we specify how data are stored.
Suppose we have $n$ distinct samples in $\mathbb{R}^p$. They are stored in a matrix of shape $(p,n)$ whose $j$-th column is a vector in $\mathbb{R}^p$ representing the $j$-th sample.
We write this matrix as $X_j[i]$ with $1\le i \le p$ and $1\le j \le n$, so that the vector $X_j$ is the $j$-th input sample.
Thus the following example represents 4 samples in $\mathbb{R}^3$

In [5]:
X =np.array([[1,2,3,4],[5,6,7,8],[9,10,11,12]])
print(X)

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]


The first operation we need is: given a matrix $A$ of shape $(q,p)$, find the matrix $Y$ of shape $(q,n)$ such that for each sample $j$, $Y_j=AX_j$. In the following example, $q=2$.

In [6]:
A=np.array([[1,2,3],[4,5,6]])
print('A=',A)
Y=A.dot(X)
print ('Y=',Y)
# You should get
# A= [[1 2 3]
# [4 5 6]]
# Y= [[ 38  44  50  56]
# [ 83  98 113 128]]

A= [[1 2 3]
 [4 5 6]]
Y= [[ 38  44  50  56]
 [ 83  98 113 128]]


In [7]:
print(A.shape)
print(X.shape)
print(Y.shape)

(2, 3)
(3, 4)
(2, 4)


We now want to add to $Y$ a vector $b \in \mathbb{R}^q$ so that for each sample $j$, $Z_j=Y_j+b$. To do this, use the following command:

In [8]:
b=np.array([1,2])
print(np.outer(b,np.ones(5)))

[[1. 1. 1. 1. 1.]
 [2. 2. 2. 2. 2.]]


You should understand the previous command and use it to compute the matrix $Z$ such that $Z_j=AX_j+b$ in the following example:

In [9]:
X =np.array([[1,1,3,4],[5,6,7,8],[9,10,11,12]])
A=np.array([[1,5,3],[4,5,6]])
b=np.array([1,3])
Z=A.dot(X) + np.outer(b,np.ones(X.shape[1]))# Fill in here!
print(Z)
# You should get
# [[ 54.  62.  72.  81.]
# [ 86.  97. 116. 131.]]

[[ 54.  62.  72.  81.]
 [ 86.  97. 116. 131.]]


# Implementing the Dense layer
A "dense" layer maps an input $X$ of size $p$ to an output $Y$ of size $q$ such that
$$Y=AX+b,$$
where $A$ is a matrix and $b$ is a vector of size $q$. The matrix $A$ and vector $b$ are the layer’s parameters.
We first focus only on the `__init__` method.

`__init__` takes two integers `nb_entree` and `nb_output` (denoted $p$ and $q$ here), corresponding to the input and output vector sizes. Store these in `self.n_entree` and `self.n_output`. The `__init__` method should also draw a random matrix $A$ of shape $(q,p)$ (stored in `self.A`) and a random vector $b$ of size $q$ (stored in `self.b`), using draws from a standard normal distribution via `numpy.random.randn`.

In [10]:
# Class test
np.random.seed(10)
import Neural as Neur
L=Neur.Dense(3,2)
print(L.n_entree)
print(L.n_sortie)
print(L.nb_params)
print('A=',L.A)
print('b=',L.b)
# You should get
# 3
# 2
# 8
# A=[[ 1.3315865   0.71527897 -1.54540029]
# [-0.00838385  0.62133597 -0.72008556]]
# b=[ 0.26551159  0.10854853  0.00429143]

3
2
8
A= [[ 1.3315865   0.71527897 -1.54540029]
 [-0.00838385  0.62133597 -0.72008556]]
b= [0.26551159 0.10854853]


We now implement `forward` for the layer class. Given a matrix $X$ of shape $(p,n)$ (where $n$ is the number of samples), `forward` computes $Y$ of shape $(q,n)$ such that $Y_j=AX_j+b$ for every $j$ from $1$ to $n$ (recall that $n$ is `X.shape[1]`). Use the approach from the previous section; in particular, do not add the vector $b$ directly in a way that breaks broadcasting across columns.

In [11]:
np.random.seed(10)
import Neural as Neur
L = Neur.Dense(3,2)
X =np.array([[1,2,3,4],[5,6,7,8],[9,10,11,12]])
print(L.forward(X))
# You should get
# [[-8.73510967 -8.23364448 -7.73217929 -7.23071411]
# [-3.2739255  -3.38105894 -3.48819237 -3.59532581]]

[[-8.73510967 -8.23364448 -7.73217929 -7.23071411]
 [-3.2739255  -3.38105894 -3.48819237 -3.59532581]]


We now turn to `get_params` and `set_params`. `get_params` returns a single long vector: the entries of $A$ followed by the entries of $b$. To flatten a matrix into a 1D array, use `ravel` in Python. `set_params` takes one long vector and fills $A$ first, then $b$, from that vector. To rebuild $A$ from the flattened coefficients, use NumPy’s `reshape`.

In [12]:
np.random.seed(10)
import Neural as Neur
L = Neur.Dense(3,2)
print('params=',L.get_params())
theta=np.random.randn(8)
L.set_params(theta)
print('theta=',theta)
print('A=',L.A)
print('b=',L.b)
# You should get
# params= [ 1.3315865   0.71527897 -1.54540029 -0.00838385  0.62133597 -0.72008556
# 0.26551159  0.10854853]
# theta= [ 0.00429143 -0.17460021  0.43302619  1.20303737 -0.96506567  1.02827408
# 0.22863013  0.44513761]
# A= [[ 0.00429143 -0.17460021  0.43302619]
# [ 1.20303737 -0.96506567  1.02827408]]
# b= [0.22863013 0.44513761]

params= [ 1.3315865   0.71527897 -1.54540029 -0.00838385  0.62133597 -0.72008556
  0.26551159  0.10854853]
theta= [ 0.00429143 -0.17460021  0.43302619  1.20303737 -0.96506567  1.02827408
  0.22863013  0.44513761]
A= [[ 0.00429143 -0.17460021  0.43302619]
 [ 1.20303737 -0.96506567  1.02827408]]
b= [0.22863013 0.44513761]


That was straightforward. Now implement backpropagation for the dense layer. Recall: if the layer maps $(p)$ to $(q)$ with $n$ samples, `backward` takes `grad_output` (denoted $g_s$ here) of shape $(q,n)$, computes gradients with respect to $A$ and $b$ (denoted $g_A$ and $g_b$, with shapes $(q,p)$ and $q$ respectively), packs them into a vector of length `self.nb_params` (as in `get_params`), and returns an input gradient `grad_entree` (denoted $g_e$) of shape $(p,n)$ for the previous layers. The formulas are:

\[g_e = A^T g_s\]
\[g_A = g_s X^T\]
\[g_b[i] = \sum_j g_s[i,j]\]
Implement `backward` for the dense layer and test your code below:

In [13]:
np.random.seed(10)
import Neural as Neur
L = Neur.Dense(3,2)
X=np.random.randn(3,4)
grad_sortie=np.random.randn(2,4)
L.forward(X)
gl,ge=L.backward(grad_sortie)
print('grad_local=',gl)
print('grad_entree=',ge)
# You should get
# grad_local= [ 3.28032607  1.23844345 -0.16801192  1.43755815  1.28044748 -2.41352966
# -1.07006308  4.29345906]
# grad_entree= [[-2.64293715 -2.33547403  0.35346419  3.16406972]
# [-0.71643766 -0.2077372   0.25191937  2.57454243]
# [ 2.24722802  1.48977695 -0.48258083 -4.69240621]]

grad_local= [ 3.28032607  1.23844345 -0.16801192  1.43755815  1.28044748 -2.41352966
 -1.07006308  4.29345906]
grad_entree= [[-2.64293715 -2.33547403  0.35346419  3.16406972]
 [-0.71643766 -0.2077372   0.25191937  2.57454243]
 [ 2.24722802  1.48977695 -0.48258083 -4.69240621]]


## Building the L2 loss layer
We now build a class for the $L_2$ loss layer. This layer has no parameters; it stores target data $D$. The forward pass computes
$$Y=\frac{1}{2}\Vert X-D\Vert^2.$$
The scalar $Y$ is typically the last layer of the network: it measures the mismatch between $X$ and the targets $D$.

For the backward pass, this layer does not need an output gradient, has no local gradient, and its input gradient is $X-D$. Implement the $L_2$ loss in a class named `Loss_L2` and run the code below:


In [14]:
np.random.seed(10)
import Neural as Neur
D=np.random.randn(3,2)
X=np.random.randn(3,2)
L = Neur.Loss_L2(D)
print(L.nb_params)
print(L.forward(X))
print(L.backward(None))
# You should get
# 0
# 3.8338361400772234
# (None, array([[-1.06607492, -0.60673045],
# [ 1.54969172, -0.16621636],
# [-0.18830978,  1.92312293]]))


0
3.8338361400772234
(None, array([[-1.06607492, -0.60673045],
       [ 1.54969172, -0.16621636],
       [-0.18830978,  1.92312293]]))


# Building the neural network
We now assemble the neural network itself. A neural network is essentially a list of layers applied in sequence.

For convenience, the `Network` class uses the same attribute names as a single layer; in fact, one can (and sometimes should) treat a whole network as one layer and stack several such “macro” layers. The class will be called `Network`.

Start with `__init__`, `set_params`, and `get_params`. `__init__` takes `list_layers`, a list of layers, and stores it in `self.list_layers`.

Compute `self.nb_params`: by definition, it is the sum of the parameter-vector sizes of every layer in the list.

Then implement `set_params` and `get_params`. `set_params` takes a vector of length `self.nb_params` and assigns the leading coefficients to the first layer, the next block to the second layer, and so on. `get_params` calls each layer’s `get_params` in order and concatenates the results into one long vector.

In [15]:
np.random.seed(10)
import Neural as Neur
D=np.random.randn(4,10)
X=np.random.randn(3,10)

L1=Neur.Dense(3,2)
L2=Neur.Arctan()
L3=Neur.Dense(2,6)
L4=Neur.Arctan()
L5=Neur.Dense(6,4)
L6=Neur.Loss_L2(D)


N=Neur.Network([L1,L2,L3,L4,L5,L6])
           
print(N.nb_params)
print('params=',N.get_params())
theta=np.random.randn(N.nb_params)
N.set_params(theta)
print(np.linalg.norm(N.get_params()-theta))
# You should get
# 54
# params= [ 0.31935642  0.4609029  -0.21578989  0.98907246  0.31475378  2.46765106
# -1.50832149  0.62060066 -1.04513254 -0.79800882  1.98508459  1.74481415
# -1.85618548 -0.2227737  -0.06584785 -2.13171211 -0.04883051  0.39334122
# 0.21726515 -1.99439377  1.10770823  0.24454398 -0.06191203 -0.75389296
# 0.71195902  0.91826915 -0.48209314  0.08958761  0.82699862 -1.95451212
# 0.11747566 -1.90745689 -0.92290926  0.46975143 -0.14436676 -0.40013835
# -0.29598385  0.84820861  0.70683045 -0.78726893  0.29294072 -0.47080725
# 2.40432561 -0.73935674 -0.31282876 -0.34888192 -0.43902624  0.14110417
# 0.27304932 -1.61857075 -0.57311336 -1.32044755  1.23620533  2.46532508]
# 0.0

54
params= [ 0.31935642  0.4609029  -0.21578989  0.98907246  0.31475378  2.46765106
 -1.50832149  0.62060066 -1.04513254 -0.79800882  1.98508459  1.74481415
 -1.85618548 -0.2227737  -0.06584785 -2.13171211 -0.04883051  0.39334122
  0.21726515 -1.99439377  1.10770823  0.24454398 -0.06191203 -0.75389296
  0.71195902  0.91826915 -0.48209314  0.08958761  0.82699862 -1.95451212
  0.11747566 -1.90745689 -0.92290926  0.46975143 -0.14436676 -0.40013835
 -0.29598385  0.84820861  0.70683045 -0.78726893  0.29294072 -0.47080725
  2.40432561 -0.73935674 -0.31282876 -0.34888192 -0.43902624  0.14110417
  0.27304932 -1.61857075 -0.57311336 -1.32044755  1.23620533  2.46532508]
0.0


We now implement the network’s `forward` and `backward`.
For `forward`, copy the input `X` into a variable `Z` and pass `Z` through each layer in order; return the final output. Copying into a temporary is important—otherwise the caller’s `X` would be overwritten by the computation.
For `backward`, propagate the gradient through the layers in reverse order. You can iterate backward with Python’s `reversed`. Concatenate all local gradients into one long vector, analogously to `get_params`.

In [16]:
np.random.seed(10)
import Neural as Neur
D=np.random.randn(4,10)
X=np.random.randn(3,10)

L1=Neur.Dense(3,2)
L2=Neur.Arctan()
L3=Neur.Dense(2,6)
L4=Neur.Arctan()
L5=Neur.Dense(6,4)
L6=Neur.Loss_L2(D)


N=Neur.Network([L1,L2,L3,L4,L5,L6])

a=N.forward(X)
b,c=N.backward(None)
print('a=',a)
print('b=',b)
print('c=',c)
# You should get
# a= 221.22208566711836
# b= [-11.30231307  24.605031    -7.02968989 -29.58355645   3.70228854
# -32.28218831 -30.89848629  24.74850639 -28.1221249    2.2068016
# -15.89016799 -14.56113369  39.54725178 -21.85809805  37.99499607
# -23.31829384  26.61064478   1.60179509  13.52040638  -4.05726552
# 13.42191519 -10.62904716  10.35644233  -3.94820181  -5.71322575
# -27.67476451  38.3148709   13.97693659  -8.54800412   0.60620022
# 19.75002508 -12.54967804  -2.11398405   7.88699083 -10.23572316
# -16.12049842 -55.35679888   5.89818847  35.3167564  -23.76380918
# 9.58230795 -20.60100053   2.13470619  12.16682271  60.20318534
# -37.40341197  -0.98368166 -13.25213968  -6.43358648  21.26910607
# -14.87907661  -0.35630809 -23.31164944  27.82326981]
# c= [[ 9.62116626e+00 -4.87640076e+00  1.06443747e-01 -3.06919940e+00
# -3.59066812e+00 -5.20284380e+00 -1.51608936e+00 -7.42926556e-01
# 1.58659223e+01  1.34791006e+01]
# [ 3.21815414e-01 -1.85582037e+00 -8.85800384e-01 -9.69444862e-01
# -1.57117837e+00 -1.78510096e+00 -2.74246971e+00  1.78176598e-02
# 3.64128666e+00  3.84196736e+00]
# [ 3.17260799e+01 -1.13094336e+01  2.85752584e+00 -7.67787992e+00
# -7.75070537e+00 -1.26159745e+01  2.58695138e+00 -2.57007417e+00
# 4.35516287e+01  3.48904212e+01]]


a= 221.22208566711845
b= [-11.30231307  24.605031    -7.02968989 -29.58355645   3.70228854
 -32.28218831 -30.89848629  24.74850639 -28.1221249    2.2068016
 -15.89016799 -14.56113369  39.54725178 -21.85809805  37.99499607
 -23.31829384  26.61064478   1.60179509  13.52040638  -4.05726552
  13.42191519 -10.62904716  10.35644233  -3.94820181  -5.71322575
 -27.67476451  38.3148709   13.97693659  -8.54800412   0.60620022
  19.75002508 -12.54967804  -2.11398405   7.88699083 -10.23572316
 -16.12049842 -55.35679888   5.89818847  35.3167564  -23.76380918
   9.58230795 -20.60100053   2.13470619  12.16682271  60.20318534
 -37.40341197  -0.98368166 -13.25213968  -6.43358648  21.26910607
 -14.87907661  -0.35630809 -23.31164944  27.82326981]
c= [[ 9.62116626e+00 -4.87640076e+00  1.06443747e-01 -3.06919940e+00
  -3.59066812e+00 -5.20284380e+00 -1.51608936e+00 -7.42926556e-01
   1.58659223e+01  1.34791006e+01]
 [ 3.21815414e-01 -1.85582037e+00 -8.85800384e-01 -9.69444862e-01
  -1.57117837e+00 -1.78510

That’s it—you are ready to build neural networks.

In [17]:
# Verify gradient for internal parameters

D=np.random.randn(6,10)
X=np.random.randn(3,10)

L1=Neur.Dense(3,2)
L2=Neur.Arctan()
L3=Neur.Dense(2,6)

L6=Neur.Loss_L2(D)

N=Neur.Network([L1,L2,L3, L6])

h = 1e-5
theta = N.get_params()
fx = N.forward(X)
d = np.random.randn(len(theta), 1)
d = d / np.linalg.norm(d)
d = d[:, 0]
theor_grad = np.dot(N.backward(None)[0], d)


N.set_params(theta + h*d)
fx_plus_h = N.forward(X)
num_grad = (fx_plus_h - fx) / h
print(theor_grad - num_grad)
print(theor_grad)
print(num_grad)

-0.1688914005641423
-0.9042754180669803
-0.735384017502838


In [18]:
# np.random.seed(10)
import Neural as Neur
D=np.random.randn(4,10)
X=np.random.randn(3,10)

L1=Neur.Dense(3,2)
L2=Neur.Arctan()
L3=Neur.Dense(2,6)
L4=Neur.Arctan()
L5=Neur.Dense(6,4)
L6=Neur.Loss_L2(D)


N=Neur.Network([L1,L2,L3,L4,L5,L6])

h = 1e-5
theta = N.get_params()
fx = N.forward(X)
d = np.random.randn(len(theta), 1)
d = d / np.linalg.norm(d)
d = d[:, 0]
theor_grad = np.dot(N.backward(None)[0], d)


N.set_params(theta + h*d)
fx_plus_h = N.forward(X)
num_grad = (fx_plus_h - fx) / h
print(theor_grad - num_grad)
print(theor_grad)
print(num_grad)

-36.81494854405567
-3.7447164233202166
33.070232120735454


In [23]:
# np.random.seed(10)
import Neural as Neur
D=np.random.randn(4,10)
X=np.random.randn(3,10)

L1=Neur.Dense(3,2)
L2=Neur.Arctan()
L3=Neur.Dense(2,6)
L4=Neur.Arctan()
L5=Neur.Dense(6,4)
L6=Neur.Loss_L2(D)


N=Neur.Network([L1,L3,L4,L5,L6])

h = 1e-5
theta = N.get_params()
fx = N.forward(X)
d = np.random.randn(len(theta), 1)
d = d / np.linalg.norm(d)
d = d[:, 0]
theor_grad = np.dot(N.backward(None)[0], d)


N.set_params(theta + h*d)
fx_plus_h = N.forward(X)
num_grad = (fx_plus_h - fx) / h
print(theor_grad - num_grad)
print(theor_grad)
print(num_grad)

0.877009771126299
-6.940624342267679
-7.817634113393978
